# Traditional AI Fundamentals - Search

**Atividade PEL202 - Search Problems - Gabriel Melo. Matrícula: 125.304-6**

For this exercise we need to solve the Canibals and Missionaries Problem using DFS and A* (A-Star) search algorithms.


In [1]:
import time
import heapq

## Solution 1: Depth First Search

In [2]:
"""
3m 3c 1b (1b significa que o barco está do lado errado "direito")
0m 0c 0b: significa que todos estão do lado oposto ao inicial, ou seja, foram subtraídos do errado.
start (3, 3, 1) end (0, 0, 0)
"""

def move_1c_1m(state):
    m, c, b = state
    
    if b == 1:
      if m == 0 or c == 0: return None
      m, c, b = m-1, c-1, 0
    elif b == 0:
      if m == 3 or c == 3: return None
      m, c, b = m+1, c+1, 1
    
    return [m, c, b]

def move_1m(state):
    m, c, b = state    
    
    if b == 1:
      if m == 0: return None
      m, c, b = m-1, c, 0
    elif b == 0:
      if m == 3: return None
      m, c, b = m+1, c, 1
    
    return [m, c, b]

def move_1c(state):
    m, c, b = state    
    
    if b == 1:
      if c == 0: return None
      m, c, b = m, c-1, 0
    elif b == 0:
      if c == 3: return None
      m, c, b = m, c+1, 1
    
    return [m, c, b]

def move_2c(state):
    m, c, b = state    
    
    if b == 1:
      if c < 2: return None
      m, c, b = m, c-2, 0
    elif b == 0:
      if c > 1: return None
      m, c, b = m, c+2, 1
    
    return [m, c, b]

def move_2m(state):
    m, c, b = state    
    
    if b == 1:
      if m < 2: return None
      m, c, b = m-2, c, 0
    elif b == 0:
      if m > 1: return None
      m, c, b = m+2, c, 1
    
    return [m, c, b]

In [3]:
def solve_cannibal_missionaries(initial_state):
    # Goal state is always the same
    goal_state = [0,0,0]

    # Stack stores: [ [current_board_list], [path_of_lists] ]
    stack = [[initial_state, []]]

    # Using a list for visited states
    visited = []

    # Name of the transition functions to be called
    transitions = [move_1c_1m, move_1m, move_1c, move_2m, move_2c]

    while stack:
        current_state, path = stack.pop()  # usar pop(0) ou deque.popleft() para fazer o BFS

        # Check if we reached the goal
        if current_state == goal_state:
            return path + [current_state]
        
        # Check if we've seen this list before
        if current_state in visited:
            continue

        # check if the number of cannibals are equal or less than missionaries
        m, c, _ = current_state
        ml, cl = 3 - m, 3 - c

        if (c > m and m > 0) or (cl > ml and ml > 0):
            continue

        # Add the current list to our history of visited lists
        visited.append(current_state)

        # Usa as funções de Transição para criar os novos estados
        for move_func in transitions:
            new_state = move_func(current_state)

            if new_state is not None:
                # We append the new list state and the updated path list
                stack.append([new_state, path + [current_state]])

    return None

In [4]:
def print_solution(solution):

    if solution:
        print(f"Solution found! Total steps: {len(solution) - 1}")
        for _, state in enumerate(solution):
            m, c, _ = state
            print(f"| {3-c}c {3-m}m | river | {c}c {m}m |")
    else:
        print("No solution found.")

In [5]:
# Example usage with the new initial state:
initial_state = [3,3,1]

start_time = time.time()
print("Timer started.")

#Chama a função Busca em Profundidade para solucionar o problema
solution_list_stack = solve_cannibal_missionaries(initial_state)

print_solution(solution_list_stack)

print(solution_list_stack)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Time elapsed: {elapsed_time:.10f} seconds")

Timer started.
Solution found! Total steps: 11
| 0c 0m | river | 3c 3m |
| 2c 0m | river | 1c 3m |
| 1c 0m | river | 2c 3m |
| 3c 0m | river | 0c 3m |
| 2c 0m | river | 1c 3m |
| 2c 2m | river | 1c 1m |
| 1c 1m | river | 2c 2m |
| 1c 3m | river | 2c 0m |
| 0c 3m | river | 3c 0m |
| 2c 3m | river | 1c 0m |
| 1c 3m | river | 2c 0m |
| 3c 3m | river | 0c 0m |
[[3, 3, 1], [3, 1, 0], [3, 2, 1], [3, 0, 0], [3, 1, 1], [1, 1, 0], [2, 2, 1], [0, 2, 0], [0, 3, 1], [0, 1, 0], [0, 2, 1], [0, 0, 0]]
Time elapsed: 0.0004677773 seconds


## Solution 2: A* (A-Star)

Using heuristics

In [6]:
"""
3m 3c 1b (1b significa que o barco está do lado errado "direito")
0m 0c 0b: significa que todos estão do lado oposto ao inicial, ou seja, foram subtraídos do errado.
start (3, 3, 1) end (0, 0, 0)
"""

def move_1c_1m(state):
    m, c, b = state
    
    if b == 1:
      if m == 0 or c == 0: return None
      m, c, b = m-1, c-1, 0
    elif b == 0:
      if m == 3 or c == 3: return None
      m, c, b = m+1, c+1, 1
    
    return (m, c, b)

def move_1m(state):
    m, c, b = state    
    
    if b == 1:
      if m == 0: return None
      m, c, b = m-1, c, 0
    elif b == 0:
      if m == 3: return None
      m, c, b = m+1, c, 1
    
    return (m, c, b)

def move_1c(state):
    m, c, b = state    
    
    if b == 1:
      if c == 0: return None
      m, c, b = m, c-1, 0
    elif b == 0:
      if c == 3: return None
      m, c, b = m, c+1, 1
    
    return (m, c, b)

def move_2c(state):
    m, c, b = state    
    
    if b == 1:
      if c < 2: return None
      m, c, b = m, c-2, 0
    elif b == 0:
      if c > 1: return None
      m, c, b = m, c+2, 1
    
    return (m, c, b)

def move_2m(state):
    m, c, b = state    
    
    if b == 1:
      if m < 2: return None
      m, c, b = m-2, c, 0
    elif b == 0:
      if m > 1: return None
      m, c, b = m+2, c, 1
    
    return (m, c, b)

In [7]:
# --- Heuristic Function ---

def heuristic(state):
    distance = 0
    m, c, b = state

    distance = (m + c) / 2
    return distance

In [8]:
def solve_cannibal_missionaries_A_star(initial_state):
    # Goal state is always the same
    goal_state = (0, 0, 0)

    # priority Queue stores heuristic value, state, path
    priority_queue = [(heuristic(initial_state), 0, initial_state,[])]

    # Using a list for visited states
    visited = set()

    # Name of the transition functions to be called
    transitions = [move_1c_1m, move_1m, move_1c, move_2m, move_2c]

    while priority_queue:
        # Pop the state with the lowest heuristic
        f_cost, g_cost, current_state, path = heapq.heappop(priority_queue)
        
        # Check if we reached the goal
        if current_state == goal_state:
            return path + [current_state]

        # Check if we've seen this list before
        if current_state in visited:
            continue

        # check if the number of cannibals are equal or less than missionaries
        m, c, _ = current_state
        ml, cl = 3 - m, 3 - c

        if (c > m and m > 0) or (cl > ml and ml > 0):
            continue

        # Add the current list to our history of visited lists
        visited.add(current_state)

        # Usa as funções de Transição para criar os novos estados
        for move_func in transitions:
            new_state = move_func(current_state)

            if new_state is not None:
                # We append the new list state and the updated path list
                new_g_cost = g_cost + 1
                new_h_cost = heuristic(new_state)
                new_f_cost = new_g_cost + new_h_cost
                heapq.heappush(priority_queue, (new_f_cost, new_g_cost, new_state, path + [current_state]))

    return None

In [9]:
# Example usage with the new initial state:
initial_state = (3, 3, 1)


start_time = time.time()
print("Timer started.")

#Chama a função Busca em Profundidade para solucionar o problema
solution_list = solve_cannibal_missionaries_A_star(initial_state)

print_solution(solution_list), print(solution_list)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Time elapsed: {elapsed_time:.10f} seconds")

Timer started.
Solution found! Total steps: 11
| 0c 0m | river | 3c 3m |
| 1c 1m | river | 2c 2m |
| 1c 0m | river | 2c 3m |
| 3c 0m | river | 0c 3m |
| 2c 0m | river | 1c 3m |
| 2c 2m | river | 1c 1m |
| 1c 1m | river | 2c 2m |
| 1c 3m | river | 2c 0m |
| 0c 3m | river | 3c 0m |
| 2c 3m | river | 1c 0m |
| 1c 3m | river | 2c 0m |
| 3c 3m | river | 0c 0m |
[(3, 3, 1), (2, 2, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (0, 2, 1), (0, 0, 0)]
Time elapsed: 0.0006003380 seconds
